In [2]:
!pip install shap


Defaulting to user installation because normal site-packages is not writeable


In [3]:
import sys
!{sys.executable} -m pip install "numpy<2.0" --force-reinstall


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.


In [4]:
import numpy as np
import pandas as pd
print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")
print("✅ Environment is healthy!")

numpy  : 1.26.4
pandas : 2.2.2
✅ Environment is healthy!


In [5]:
pip install "shap==0.46.0"

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [6]:
import sys
sys.path.append(r'C:\Users\win10\AppData\Roaming\Python\Python312\site-packages')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print(f"✅ SHAP version : {shap.__version__}")
print(f"✅ NumPy version : {np.__version__}")
print("✅ Imports done")

✅ SHAP version : 0.46.0
✅ NumPy version : 1.26.4
✅ Imports done


In [7]:
xgb_model = joblib.load('xgb_model.pkl')

X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv').squeeze()

print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")
print("✅ Model and data loaded")

X_test shape : (68366, 18)
y_test shape : (68366,)
✅ Model and data loaded


In [8]:
import json

# Extract and patch the booster model config
booster = xgb_model.get_booster()
config = json.loads(booster.save_config())

# Fix the base_score format
config['learner']['learner_model_param']['base_score'] = '5E-1'
config['learner']['learner_model_param']['base_score'] = str(0.5)

# Save and reload the model as JSON to fix compatibility
booster.save_model('xgb_temp.json')

import xgboost as xgb
booster2 = xgb.Booster()
booster2.load_model('xgb_temp.json')

# Manually patch base_score in config
cfg = json.loads(booster2.save_config())
cfg['learner']['learner_model_param']['base_score'] = '0.5'
booster2.load_config(json.dumps(cfg))

explainer = shap.TreeExplainer(booster2)

sample_size = min(2000, len(X_test))
X_sample = X_test.sample(sample_size, random_state=42).reset_index(drop=True)
y_sample = y_test.loc[X_test.sample(sample_size, random_state=42).index].reset_index(drop=True)

print(f"Computing SHAP values on {sample_size} samples...")
shap_values = explainer.shap_values(X_sample)
print(f"SHAP values shape : {shap_values.shape}")
print("✅ SHAP values computed")

Computing SHAP values on 2000 samples...
SHAP values shape : (2000, 18)
✅ SHAP values computed


In [11]:
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title("Top Features — Global Importance (SHAP)")
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved shap_summary_bar.png")

✅ Saved shap_summary_bar.png


In [12]:
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP Beeswarm — Feature Impact Direction")
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved shap_beeswarm.png")

✅ Saved shap_beeswarm.png


In [13]:
fraud_indices = np.where(y_sample.values == 1)[0]

if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[0]
    print(f"Explaining fraud transaction at index {fraud_idx}")

    expected_val = explainer.expected_value
    if isinstance(expected_val, (list, np.ndarray)):
        expected_val = float(expected_val[1])
        sv = shap_values[fraud_idx]
    else:
        expected_val = float(expected_val)
        sv = shap_values[fraud_idx]

    shap.waterfall_plot(
        shap.Explanation(
            values=sv,
            base_values=expected_val,
            data=X_sample.iloc[fraud_idx].values,
            feature_names=X_sample.columns.tolist()
        ),
        show=False
    )
    plt.tight_layout()
    plt.savefig('shap_waterfall_fraud.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("✅ Saved shap_waterfall_fraud.png")
else:
    print("⚠️ No fraud in sample — increase sample_size to 5000 in Cell 4")

Explaining fraud transaction at index 1311
✅ Saved shap_waterfall_fraud.png


In [14]:
shap_df = pd.DataFrame(shap_values, columns=X_sample.columns)
shap_df.to_csv('shap_values.csv', index=False)

print("✅ Saved shap_values.csv")
print(f"   File size : {os.path.getsize('shap_values.csv') / 1024:.1f} KB")

✅ Saved shap_values.csv
   File size : 389.7 KB


In [15]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done")

✅ Imports done


In [20]:
# Fix: Load GNN scores and align to X_test indices
gnn_raw = pd.read_csv('gnn_scores.csv')

# X_test index corresponds to node_id — filter GNN scores to match
X_test_indices = X_test.index.values  # original row indices

gnn_aligned = gnn_raw[gnn_raw['node_id'].isin(X_test_indices)].set_index('node_id')
gnn_aligned = gnn_aligned.loc[X_test_indices]  # reorder to match X_test exactly

print(f"X_test shape          : {X_test.shape}")
print(f"gnn_aligned shape     : {gnn_aligned.shape}")
print(f"Indices match         : {len(gnn_aligned) == len(X_test)}")
print(gnn_aligned.head())
print("✅ GNN scores aligned")

X_test shape          : (68366, 18)
gnn_aligned shape     : (68366, 3)
Indices match         : True
         fraud_prob  fraud_pred  actual_label
node_id                                      
0          0.002439           0             0
1          0.002376           0             0
2          0.004864           0             0
3          0.002258           0             0
4          0.002586           0             0
✅ GNN scores aligned


In [21]:
# XGBoost probabilities
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Isolation Forest scores
iso_raw = iso_model.decision_function(X_test)
iso_probs = (iso_raw - iso_raw.min()) / (iso_raw.max() - iso_raw.min())
iso_probs = 1 - iso_probs

# GNN scores — use aligned fraud_prob column
gnn_probs = gnn_aligned['fraud_prob'].values
gnn_probs = (gnn_probs - gnn_probs.min()) / (gnn_probs.max() - gnn_probs.min())

print(f"XGB probs shape  : {xgb_probs.shape}")
print(f"ISO probs shape  : {iso_probs.shape}")
print(f"GNN probs shape  : {gnn_probs.shape}")
print(f"XGB — min: {xgb_probs.min():.3f}, max: {xgb_probs.max():.3f}")
print(f"ISO — min: {iso_probs.min():.3f}, max: {iso_probs.max():.3f}")
print(f"GNN — min: {gnn_probs.min():.3f}, max: {gnn_probs.max():.3f}")
print("✅ All shapes match — ready for ensemble")

XGB probs shape  : (68366,)
ISO probs shape  : (68366,)
GNN probs shape  : (68366,)
XGB — min: 0.000, max: 0.016
ISO — min: 0.000, max: 1.000
GNN — min: 0.000, max: 1.000
✅ All shapes match — ready for ensemble


In [22]:
# Weights: XGBoost is strongest, GNN second, IsoForest third
w_xgb, w_gnn, w_iso = 0.5, 0.3, 0.2

ensemble_score = (w_xgb * xgb_probs) + (w_gnn * gnn_probs) + (w_iso * iso_probs)

# Final prediction at threshold 0.5
threshold = 0.5
ensemble_pred = (ensemble_score >= threshold).astype(int)

print(f"Ensemble fraud detected : {ensemble_pred.sum()} / {len(ensemble_pred)}")
print(f"Fraud rate              : {ensemble_pred.mean()*100:.2f}%")
print("✅ Ensemble predictions done")

Ensemble fraud detected : 0 / 68366
Fraud rate              : 0.00%
✅ Ensemble predictions done


In [23]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score)

auc = roc_auc_score(y_test, ensemble_score)
ap  = average_precision_score(y_test, ensemble_score)

print("=" * 50)
print("        ENSEMBLE MODEL — EVALUATION")
print("=" * 50)
print(f"  ROC-AUC Score      : {auc:.4f}")
print(f"  Avg Precision (AP) : {ap:.4f}")
print("=" * 50)
print(classification_report(y_test, ensemble_pred, target_names=['Legit', 'Fraud']))
print("✅ Metrics computed")

        ENSEMBLE MODEL — EVALUATION
  ROC-AUC Score      : 0.4725
  Avg Precision (AP) : 0.0007
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     68327
       Fraud       0.00      0.00      0.00        39

    accuracy                           1.00     68366
   macro avg       0.50      0.50      0.50     68366
weighted avg       1.00      1.00      1.00     68366

✅ Metrics computed


In [25]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, ensemble_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Ensemble Model — Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved confusion_matrix.png")

✅ Saved confusion_matrix.png


In [26]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, ensemble_score)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Ensemble ROC (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Ensemble Fraud Detector')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved roc_curve.png")

✅ Saved roc_curve.png


In [27]:
plt.figure(figsize=(8, 5))
plt.hist(ensemble_score[y_test == 0], bins=50, alpha=0.6, color='steelblue', label='Legit')
plt.hist(ensemble_score[y_test == 1], bins=50, alpha=0.6, color='crimson',   label='Fraud')
plt.xlabel('Ensemble Fraud Score')
plt.ylabel('Count')
plt.title('Fraud Score Distribution — Ensemble Model')
plt.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved score_distribution.png")

✅ Saved score_distribution.png


In [28]:
results_df = pd.DataFrame({
    'xgb_score'      : xgb_probs,
    'iso_score'      : iso_probs,
    'gnn_score'      : gnn_probs,
    'ensemble_score' : ensemble_score,
    'ensemble_pred'  : ensemble_pred,
    'true_label'     : y_test.values
})
results_df.to_csv('ensemble_results.csv', index=False)

print("✅ Saved ensemble_results.csv")
print(f"   Shape : {results_df.shape}")

✅ Saved ensemble_results.csv
   Shape : (68366, 6)


In [29]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done")

✅ Imports done


In [30]:
# Load ensemble results
results_df = pd.read_csv('ensemble_results.csv')

# Load SHAP values
shap_df = pd.read_csv('shap_values.csv')

# Load test data
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv').squeeze()

print(f"Ensemble results shape : {results_df.shape}")
print(f"SHAP values shape      : {shap_df.shape}")
print(f"X_test shape           : {X_test.shape}")
print("✅ All results loaded")

Ensemble results shape : (68366, 6)
SHAP values shape      : (2000, 18)
X_test shape           : (68366, 18)
✅ All results loaded


In [31]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)

y_true  = results_df['true_label']
y_score = results_df['ensemble_score']
y_pred  = results_df['ensemble_pred']

auc       = roc_auc_score(y_true, y_score)
ap        = average_precision_score(y_true, y_score)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
fraud_detected = y_pred.sum()
total          = len(y_pred)
fraud_rate     = y_pred.mean() * 100

print("=" * 50)
print("       FINAL MODEL METRICS SUMMARY")
print("=" * 50)
print(f"  ROC-AUC Score      : {auc:.4f}")
print(f"  Avg Precision (AP) : {ap:.4f}")
print(f"  Precision          : {precision:.4f}")
print(f"  Recall             : {recall:.4f}")
print(f"  F1 Score           : {f1:.4f}")
print(f"  Fraud Detected     : {fraud_detected} / {total}")
print(f"  Fraud Rate         : {fraud_rate:.2f}%")
print("=" * 50)
print("✅ Metrics computed")

       FINAL MODEL METRICS SUMMARY
  ROC-AUC Score      : 0.4725
  Avg Precision (AP) : 0.0007
  Precision          : 0.0000
  Recall             : 0.0000
  F1 Score           : 0.0000
  Fraud Detected     : 0 / 68366
  Fraud Rate         : 0.00%
✅ Metrics computed


In [32]:
# Get mean absolute SHAP value per feature
shap_importance = shap_df.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(8, 6))
shap_importance.head(10).plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Mean |SHAP Value|')
plt.title('Top 10 Most Important Features — SHAP')
plt.tight_layout()
plt.savefig('top10_features.png', dpi=150, bbox_inches='tight')
plt.close()

print("Top 10 Features:")
for i, (feat, val) in enumerate(shap_importance.head(10).items(), 1):
    print(f"  {i:>2}. {feat:<30} {val:.4f}")
print("✅ Saved top10_features.png")

Top 10 Features:
   1. orig_emptied                   5.6782
   2. balance_diff_orig              0.7932
   3. balance_diff_dest              0.6795
   4. type_CASH_OUT                  0.6786
   5. step                           0.5422
   6. amount_vs_orig_balance         0.4994
   7. newbalanceDest                 0.4972
   8. type_PAYMENT                   0.3627
   9. amount                         0.3511
  10. log_amount                     0.3352
✅ Saved top10_features.png


In [33]:
# Compare individual models vs ensemble
xgb_auc = roc_auc_score(y_true, results_df['xgb_score'])
iso_auc  = roc_auc_score(y_true, results_df['iso_score'])
gnn_auc  = roc_auc_score(y_true, results_df['gnn_score'])
ens_auc  = roc_auc_score(y_true, results_df['ensemble_score'])

xgb_ap = average_precision_score(y_true, results_df['xgb_score'])
iso_ap  = average_precision_score(y_true, results_df['iso_score'])
gnn_ap  = average_precision_score(y_true, results_df['gnn_score'])
ens_ap  = average_precision_score(y_true, results_df['ensemble_score'])

comparison_df = pd.DataFrame({
    'Model'     : ['XGBoost', 'Isolation Forest', 'GNN', 'Ensemble'],
    'ROC-AUC'   : [xgb_auc, iso_auc, gnn_auc, ens_auc],
    'Avg Precision': [xgb_ap, iso_ap, gnn_ap, ens_ap]
})
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))
comparison_df.to_csv('model_comparison.csv', index=False)
print("\n✅ Saved model_comparison.csv")

           Model  ROC-AUC  Avg Precision
         XGBoost 0.657557       0.000905
             GNN 0.571217       0.000884
        Ensemble 0.472541       0.000715
Isolation Forest 0.472227       0.000717

✅ Saved model_comparison.csv


In [34]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']

axes[0].bar(comparison_df['Model'], comparison_df['ROC-AUC'], color=colors)
axes[0].set_title('ROC-AUC by Model')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_ylim(0, 1)
for i, v in enumerate(comparison_df['ROC-AUC']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)

axes[1].bar(comparison_df['Model'], comparison_df['Avg Precision'], color=colors)
axes[1].set_title('Average Precision by Model')
axes[1].set_ylabel('Avg Precision')
axes[1].set_ylim(0, 1)
for i, v in enumerate(comparison_df['Avg Precision']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Model Comparison — Fraud Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved model_comparison.png")

✅ Saved model_comparison.png


In [35]:
submission = {
    "project"    : "FusionX Dizzy Hackers Mega Hackathon 2026",
    "track"      : "AI/ML & Data Science",
    "title"      : "Fraud Detection — Ensemble of XGBoost + GNN + Isolation Forest",
    "models"     : {
        "XGBoost"          : {"weight": 0.5, "roc_auc": round(xgb_auc, 4)},
        "GNN"              : {"weight": 0.3, "roc_auc": round(gnn_auc, 4)},
        "Isolation Forest" : {"weight": 0.2, "roc_auc": round(iso_auc, 4)}
    },
    "ensemble_metrics" : {
        "roc_auc"       : round(auc, 4),
        "avg_precision" : round(ap, 4),
        "precision"     : round(precision, 4),
        "recall"        : round(recall, 4),
        "f1_score"      : round(f1, 4),
        "fraud_detected": int(fraud_detected),
        "total_samples" : int(total)
    },
    "explainability" : "SHAP TreeExplainer — feature-level explanations for every prediction",
    "files_submitted": [
        "xgb_model.pkl", "iso_model.pkl", "gnn_best.pt",
        "ensemble_results.csv", "shap_values.csv",
        "model_comparison.csv", "model_comparison.png",
        "roc_curve.png", "confusion_matrix.png",
        "score_distribution.png", "shap_summary_bar.png",
        "shap_beeswarm.png", "shap_waterfall_fraud.png",
        "top10_features.png"
    ]
}

with open('submission.json', 'w') as f:
    json.dump(submission, f, indent=4)

print("✅ Saved submission.json")
print(json.dumps(submission, indent=4))

✅ Saved submission.json
{
    "project": "FusionX Dizzy Hackers Mega Hackathon 2026",
    "track": "AI/ML & Data Science",
    "title": "Fraud Detection \u2014 Ensemble of XGBoost + GNN + Isolation Forest",
    "models": {
        "XGBoost": {
            "weight": 0.5,
            "roc_auc": 0.6576
        },
        "GNN": {
            "weight": 0.3,
            "roc_auc": 0.5712
        },
        "Isolation Forest": {
            "weight": 0.2,
            "roc_auc": 0.4722
        }
    },
    "ensemble_metrics": {
        "roc_auc": 0.4725,
        "avg_precision": 0.0007,
        "precision": 0.0,
        "recall": 0.0,
        "f1_score": 0.0,
        "fraud_detected": 0,
        "total_samples": 68366
    },
    "explainability": "SHAP TreeExplainer \u2014 feature-level explanations for every prediction",
    "files_submitted": [
        "xgb_model.pkl",
        "iso_model.pkl",
        "gnn_best.pt",
        "ensemble_results.csv",
        "shap_values.csv",
        "model_c

In [36]:
required_files = [
    'xgb_model.pkl', 'iso_model.pkl', 'gnn_best.pt',
    'ensemble_results.csv', 'shap_values.csv',
    'model_comparison.csv', 'model_comparison.png',
    'roc_curve.png', 'confusion_matrix.png',
    'score_distribution.png', 'shap_summary_bar.png',
    'shap_beeswarm.png', 'shap_waterfall_fraud.png',
    'top10_features.png', 'submission.json'
]

print("📁 FILE CHECKLIST:")
all_ok = True
for f in required_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✅ {f:<40} ({size:.1f} KB)")
    else:
        print(f"  ❌ MISSING: {f}")
        all_ok = False

print()
if all_ok:
    print("🎉 ALL FILES PRESENT — READY TO SUBMIT!")
else:
    print("⚠️  Some files are missing — check above!")

📁 FILE CHECKLIST:
  ✅ xgb_model.pkl                            (236.9 KB)
  ✅ iso_model.pkl                            (1032.2 KB)
  ✅ gnn_best.pt                              (48.1 KB)
  ✅ ensemble_results.csv                     (4891.2 KB)
  ✅ shap_values.csv                          (389.7 KB)
  ✅ model_comparison.csv                     (0.2 KB)
  ✅ model_comparison.png                     (53.6 KB)
  ✅ roc_curve.png                            (54.6 KB)
  ✅ confusion_matrix.png                     (37.6 KB)
  ✅ score_distribution.png                   (37.0 KB)
  ✅ shap_summary_bar.png                     (102.2 KB)
  ✅ shap_beeswarm.png                        (137.5 KB)
  ✅ shap_waterfall_fraud.png                 (99.8 KB)
  ✅ top10_features.png                       (51.0 KB)
  ✅ submission.json                          (1.3 KB)

🎉 ALL FILES PRESENT — READY TO SUBMIT!


In [2]:
import pandas as pd

X_test = pd.read_csv('X_test.csv')

In [3]:
print(X_test.columns.tolist())

['step', 'amount', 'log_amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'orig_emptied', 'dest_unchanged', 'amount_vs_orig_balance', 'balance_diff_orig', 'balance_diff_dest', 'amount_mismatch']
